# 🗺️ Módulo 09 - Notebook 02: GeoDataFrames, CRS y Shapefiles

## 📁 Manejo avanzado de datos geoespaciales

**Libro:** Saliendo de lo Pandito  
**Módulo:** 09 - Analítica Geoespacial GeoPandas  
**Duración estimada:** 65 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** GeoDataFrames desde coordenadas  
✅ **Entender** Sistemas de Referencia de Coordenadas (CRS)  
✅ **Transformar** entre diferentes CRS  
✅ **Leer** y escribir archivos Shapefile y GeoJSON  
✅ **Combinar** datos espaciales con datos de negocio

---

## 📋 Pre-requisitos

* ✅ Notebook 09_01 completado (Introducción a GeoPandas)
* ✅ Conocimiento de geometrías básicas
* ✅ Familiaridad con formatos de archivo geoespaciales

---

## 📚 Contenido

1. Creación de GeoDataFrames
2. Sistemas de Coordenadas (CRS)
3. Transformaciones de CRS
4. Lectura de Shapefiles
5. Exportación a GeoJSON
6. Caso Integrador: Análisis Territorial

---

## 💡 Por qué importa

**Trabajar con datos espaciales reales:**

* 📁 **Shapefiles:** Formato estándar de GIS
* 🌐 **GeoJSON:** Formato web moderno
* 🗺️ **CRS:** Correcta interpretación de coordenadas
* 🔄 **Transformaciones:** Integrar datos de múltiples fuentes

**La base de todo análisis geoespacial profesional**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (con coordenadas)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Extraer ubicaciones únicas de sucursales
    df_sucursales = df_ventas[['sucursal_id', 'sucursal_nombre', 'zona', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)
    
    # Agregar ventas totales por sucursal
    ventas_totales = df_ventas.groupby('sucursal_id')['ventas'].sum().reset_index()
    ventas_totales.columns = ['sucursal_id', 'ventas_totales']
    df_sucursales = df_sucursales.merge(ventas_totales, on='sucursal_id')
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros ventas: {len(df_ventas):,}")
    print(f"   🏪 Sucursales: {len(df_sucursales)}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n🗺️ Datos espaciales:")
    print(f"   • Coordenadas WGS84 (lat, lon)")
    print(f"   • Listo para crear GeoDataFrame")
    print(f"   • Ventas totales por sucursal disponibles")
    
    print(f"\n🎯 Este notebook creará GeoDataFrames con datos REALES")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    df_sucursales = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 GeoDataFrames, CRS y Formatos de Archivo

### 🗺️ ¿Qué es un GeoDataFrame?

Un **GeoDataFrame** es un DataFrame de Pandas con una columna especial de geometrías.

**Estructura:**
```
  sucursal    ventas                    geometry        CRS
0  Centro     100000  POINT (-68.85 -32.89)  EPSG:4326
1  Norte      150000  POINT (-68.83 -32.87)  EPSG:4326
```

---

### 🌐 Sistemas de Coordenadas (CRS)

**CRS** (Coordinate Reference System) define cómo interpretar las coordenadas.

#### **Los 2 CRS más importantes:**

**1️⃣ WGS84 (EPSG:4326)**
* **Unidades:** Grados (latitud, longitud)
* **Uso:** GPS, Google Maps, datos raw
* **Rango:** Lat: -90 a +90, Lon: -180 a +180

**2️⃣ Web Mercator (EPSG:3857)**
* **Unidades:** Metros
* **Uso:** Mapas web (Google, OSM)
* **Ventaja:** Distancias en metros (fácil calcular áreas/distancias)

---

### 🔄 Transformación de CRS

**¿Por qué transformar?**
* Calcular distancias en metros
* Calcular áreas en km²
* Integrar datos de diferentes fuentes

**Sintaxis:**
```python
# De WGS84 (grados) a Web Mercator (metros)
gdf_metros = gdf.to_crs(epsg=3857)

# De vuelta a WGS84
gdf_grados = gdf_metros.to_crs(epsg=4326)
```

---

### 📁 Formatos de Archivo

#### **Shapefile (.shp)**
* Formato estándar de GIS (1990s)
* Múltiples archivos (.shp, .shx, .dbf, .prj)
* Limitaciones: nombres de columna max 10 chars

```python
# Leer
gdf = gpd.read_file('datos.shp')

# Escribir
gdf.to_file('salida.shp')
```

---

#### **GeoJSON (.geojson)**
* Formato moderno, basado en JSON
* Un solo archivo
* Compatible con web

```python
# Leer
gdf = gpd.read_file('datos.geojson')

# Escribir
gdf.to_file('salida.geojson', driver='GeoJSON')
```

---

### 🛠️ Creación de GeoDataFrame

**Desde coordenadas:**
```python
import geopandas as gpd
from shapely.geometry import Point

# DataFrame con lat/lon
df = pd.DataFrame({
    'nombre': ['Sucursal A', 'Sucursal B'],
    'lat': [-32.89, -32.87],
    'lon': [-68.85, -68.83]
})

# Crear geometrías
geometry = [Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])]

# Crear GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
```

---

### 💡 Regla de Oro

👉 **Siempre especifica el CRS al crear un GeoDataFrame**  
👉 **WGS84 (EPSG:4326)** para datos GPS/coordenadas  
👉 **Web Mercator (EPSG:3857)** para cálculos métricos

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🗺️ GEODATAFRAMES, CRS Y SHAPEFILES")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    import geopandas as gpd
    from shapely.geometry import Point, LineString, Polygon
    print(f"Versión de GeoPandas: {gpd.__version__}")
except ImportError:
    print("⚠️  GeoPandas no instalado. Ejecuta: %pip install geopandas")

print("\n🎯 En este notebook aprenderás:")
print("  • Crear GeoDataFrames desde coordenadas")
print("  • Sistemas de coordenadas (CRS)")
print("  • Transformaciones: WGS84 ↔ Web Mercator")
print("  • Leer/escribir Shapefiles y GeoJSON")

print("\n📖 Métodos clave:")
print("  - gpd.GeoDataFrame(df, geometry=points, crs='EPSG:4326')")
print("  - gdf.to_crs(epsg=3857)  # Transformar CRS")
print("  - gpd.read_file('archivo.shp')  # Leer")
print("  - gdf.to_file('salida.geojson', driver='GeoJSON')")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# 🎯 OPCIONAL: Exportar datos georeferenciados a Shapefile/GeoJSON

# Descomentar para usar datos reales:
"""
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("💾 Cargando datos georeferenciados desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    # Obtener sucursales únicas
    df_sucursales = df_ventas[[
        'sucursal_id', 'sucursal_nombre', 'lat', 'lon', 'zona'
    ]].drop_duplicates()
    
    # Crear GeoDataFrame en CRS WGS84 (EPSG:4326)
    geometry = [Point(xy) for xy in zip(df_sucursales['lon'], df_sucursales['lat'])]
    gdf = gpd.GeoDataFrame(df_suculsales, geometry=geometry, crs='EPSG:4326')
    
    print(f"✅ GeoDataFrame creado:")
    print(f"   • CRS original: {gdf.crs} (WGS84 - GPS)")
    
    # Reproyectar a sistema métrico para Argentina (POSGAR 94)
    gdf_posgar = gdf.to_crs('EPSG:22185')
    print(f"   • CRS reproyectado: {gdf_posgar.crs} (POSGAR 94 - Argentina)")
    
    print(f"\n💡 Operaciones disponibles:")
    print("\n1️⃣ Exportar a Shapefile:")
    print("   gdf.to_file('/dbfs/tmp/sucursales.shp')")
    
    print("\n2️⃣ Exportar a GeoJSON:")
    print("   gdf.to_file('/dbfs/tmp/sucursales.geojson', driver='GeoJSON')")
    
    print("\n3️⃣ Exportar a GeoParquet (recomendado):")
    print("   gdf.to_parquet('/dbfs/tmp/sucursales.geoparquet')")
    
    print(f"\n🗺️ Sistemas de Coordenadas Relevantes para Argentina:")
    print("   • EPSG:4326 (WGS84) - Coordenadas GPS globales")
    print("   • EPSG:22185 (POSGAR 94 Zona 2) - Mendoza, San Juan, La Rioja")
    print("   • EPSG:22183 (POSGAR 94 Zona 4) - Buenos Aires, CABA")
    print("   • EPSG:5347 (POSGAR 2007) - Sistema unificado Argentina")
    
    display(gdf.head())
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del notebook 09_02

### ✅ Lo que aprendiste

1. **Creación de GeoDataFrames:**
   - `gpd.GeoDataFrame(df, geometry=points, crs='EPSG:4326')`
   - Convierte un DataFrame plano en un DataFrame espacial
   - La columna `geometry` contiene las geometrías Shapely

2. **Sistemas de Referencia de Coordenadas (CRS):**
   - `EPSG:4326` (WGS84): grados lat/lon, estándar GPS
   - `EPSG:3857` (Web Mercator): metros, mapas web
   - `EPSG:22185` (POSGAR 94): metros, Argentina
   - Sin CRS, las operaciones espaciales no tienen significado

3. **Transformaciones de CRS:**
   - `gdf.to_crs(epsg=3857)` convierte entre sistemas
   - Para distancias/áreas reales: proyectar a CRS en metros
   - WGS84 → POSGAR para cálculos en Argentina

4. **Lectura y escritura de archivos:**
   - `gpd.read_file('datos.shp')` — leer Shapefile
   - `gdf.to_file('salida.geojson', driver='GeoJSON')` — escribir GeoJSON
   - Shapefile: estándar GIS, múltiples archivos, columnas ≤10 chars
   - GeoJSON: un solo archivo, compatible web

5. **Caso integrador:**
   - GeoDataFrames con datos reales de Los Andes Market
   - Sucursales georeferenciadas con ventas totales
   - Exportación a GeoJSON para visualización web

---

### 🎯 Reglas de Oro

👉 **Regla #1: Siempre especificar CRS al crear el GeoDataFrame**
```python
# MALO: sin CRS, operaciones espaciales fallan
gdf = gpd.GeoDataFrame(df, geometry=points)

# BUENO: CRS desde el inicio
gdf = gpd.GeoDataFrame(df, geometry=points, crs='EPSG:4326')
```

👉 **Regla #2: Para distancias y áreas, proyectar a metros**
```python
# MALO: distancia en grados (sin significado físico)
dist = gdf_wgs84.geometry.iloc[0].distance(gdf_wgs84.geometry.iloc[1])

# BUENO: proyectar a CRS métrico antes de medir
gdf_metros = gdf_wgs84.to_crs('EPSG:22185')  # POSGAR Argentina
dist = gdf_metros.geometry.iloc[0].distance(gdf_metros.geometry.iloc[1])
# Resultado en metros
```

👉 **Regla #3: GeoJSON para web, Shapefile para GIS desktop**
```python
# GeoJSON: un archivo, compatible con Leaflet/Mapbox
gdf.to_file('salida.geojson', driver='GeoJSON')

# Shapefile: estándar GIS, pero columnas máximo 10 caracteres
gdf.to_file('salida.shp')
# ⚠️ 'sucursal_nombre' se trunca a 'sucursal_n'
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Crear GeoDataFrame desde lat/lon | `gpd.GeoDataFrame(df, geometry=points, crs='EPSG:4326')` |
| Coordenadas GPS (lat/lon) | `crs='EPSG:4326'` (WGS84) |
| Calcular distancias reales | `gdf.to_crs('EPSG:22185').distance(...)` |
| Calcular áreas reales | `gdf.to_crs('EPSG:22185').area` |
| Mapas web (tiles, Leaflet) | `gdf.to_crs('EPSG:3857')` |
| Exportar para GIS desktop | `gdf.to_file('salida.shp')` |
| Exportar para web | `gdf.to_file('salida.geojson', driver='GeoJSON')` |
| Integrar datos de múltiples fuentes | `gpd.sjoin(gdf1, gdf2)` tras unificar CRS |
| Argentina (Mendoza) | `EPSG:22185` (POSGAR 94 Zona 2) |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🗺️ ¡GeoDataFrames, CRS y Shapefiles dominados!</h3>
  <p><i>"El CRS correcto es la diferencia entre una distancia precisa y un número sin sentido."</i></p>
</div>